# Web Scraping Notebook/API Development

In [1]:
import requests
import re
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import time
import sys
from selenium import webdriver
from selenium.webdriver.common.by import By

In [2]:
url = 'https://ll.thespacedevs.com/2.2.0/launch/?limit=100'

In [3]:
# Pull about 1000 launches:
launches = []

headers = {"User-Agent": "Stat386RocketDataset (student project)"}

while url and len(launches) < 1000:

    response = requests.get(url, headers=headers)

    if response.status_code != 200:
        print("Error:", response.status_code)
        print(response.text)
        break

    data = response.json()

    for launch in data.get("results", []):
        launches.append({
            "name": launch["name"],
            "launch_date": launch["net"],
            "provider": launch["launch_service_provider"]["name"],
            "rocket": launch["rocket"]["configuration"]["name"],
            "launch_site": launch["pad"]["name"],
            "location": launch["pad"]["location"]["name"],
            "status": launch["status"]["name"]
        })

    print("Collected:", len(launches))

    url = data.get("next")

    time.sleep(2)   # important for avoiding throttling

df = pd.DataFrame(launches)

print(df.shape)
df.head()

Collected: 100
Collected: 200
Collected: 300
Collected: 400
Collected: 500
Error: 429
{"detail":"Request was throttled. Expected available in 3267 seconds."}
(500, 7)


,name,launch_date,provider,rocket,launch_site,location,status
0,Sputnik 8K74PS | Sputnik 1,1957-10-04T19:28:34Z,Soviet Space Program,Sputnik 8K74PS,1/5,"Baikonur Cosmodrome, Republic of Kazakhstan",Launch Successful
1,Sputnik 8K74PS | Sputnik 2,1957-11-03T02:30:00Z,Soviet Space Program,Sputnik 8K74PS,1/5,"Baikonur Cosmodrome, Republic of Kazakhstan",Launch Successful
2,Vanguard | Vanguard,1957-12-06T16:44:35Z,US Navy,Vanguard,Launch Complex 18A,"Cape Canaveral SFS, FL, USA",Launch Failure
3,Juno-I | Explorer 1,1958-02-01T03:47:56Z,Army Ballistic Missile Agency,Juno-I,Launch Complex 26A,"Cape Canaveral SFS, FL, USA",Launch Successful
4,Vanguard | Vanguard,1958-02-05T07:33:00Z,US Navy,Vanguard,Launch Complex 18A,"Cape Canaveral SFS, FL, USA",Launch Failure


In [4]:
# Save dataset to csv
df.to_csv("rocket_launches_raw.csv", index=False)

In [6]:
# Convert date column
df["launch_date"] = pd.to_datetime(df["launch_date"])
df["launch_year"] = df["launch_date"].dt.year

In [ ]:
# Save to cleaned csv
df.to_csv("rocket_launches_cleaned.csv", index=False)